# Problem

使用 MNIST 建立卷積神經網路進行手寫數字分類，同時讓 validation 只取自訓練資料，保留 test split 到訓練完成後才評估。


# Method

- 影像維持 28×28 灰階格式並正規化至 0–1。
- 模型保留原本的兩個 Conv2D、Batch Normalization、Max Pooling、Dropout 與 Dense 分類層。
- 固定隨機種子為 2026，並以訓練資料的 10% 作為 validation split。
- 模型訓練與選擇完成後，才使用保留的 test split 做一次最終評估。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.datasets import mnist

# 讓模型初始化與資料切分可重現
tf.keras.utils.set_random_seed(2026)


## Data preparation

載入 MNIST，保留原本的 reshape、正規化與 one-hot encoding 步驟。


In [ ]:
# 1. 數據準備 (MNIST 數字辨識)
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# 標準化與重塑格式
x_train = x_train.reshape(60000, 28, 28, 1) / 255
x_test = x_test.reshape(10000, 28, 28, 1) / 255
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)


## Model definition

保留原始 CNN 架構與 Adam 設定，只將程式拆成較容易閱讀的責任區塊。


In [ ]:
# 2. 建立 CNN 模型
model = Sequential()

# --- BatchNormalization (批次標準化) ---
# 協助穩定每一層的特徵分布。
model.add(Conv2D(32, (3,3), padding='same', input_shape=(28,28,1), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Conv2D(64, (3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))

# --- Dropout (隨機失活) ---
# 隨機停用部分神經元，以降低過擬合風險。
model.add(Dropout(0.25))
model.add(Flatten())

# 全連接分類層
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(10, activation='softmax'))

# 3. 組裝模型 (Adam 優化器與分類損失函數)
model.compile(loss='categorical_crossentropy',
              optimizer=Adam(learning_rate=0.001),
              metrics=['accuracy'])


## Training

以 training-only validation split 訓練；本階段不讀取 test split 作為 validation data。


In [ ]:
# 4. 開始訓練 (batch_size=256)
model.fit(x_train, y_train,
          batch_size=256,
          epochs=6,
          validation_split=0.1)


# Results

執行 Notebook 時會產生訓練與 validation 指標，並在訓練完成後評估保留的 test split。公開版本不保存執行輸出，因此不宣稱特定準確率；結果需在實際執行環境中重新產生。


In [ ]:
# 5. 評估與預測結果
loss, acc = model.evaluate(x_test, y_test)
print(f'\n 測試正確率：{acc*100:.2f}%')


## Prediction demo

使用已完成訓練的模型預測 test split，並顯示固定索引的影像作為流程示例。


In [ ]:
# 預測介面
def my_predict(n):
    y_predict = np.argmax(model.predict(x_test), axis=-1)
    print(f' 我的 CNN 預測結果是：{y_predict[n]}')
    plt.imshow(x_test[n].reshape(28,28), cmap='Greys')

# 測試第 100 號圖片
my_predict(100)


# Limitations

- MNIST 與真實手寫輸入的分布可能不同，測試結果不能直接代表部署情境。
- `validation_split=0.1` 取自訓練資料；test split 不參與訓練期間的模型調整。
- 範例預測固定顯示 test split 的第 100 張影像，僅用於說明推論流程。
